In [1]:

from collections import defaultdict

class FiveGramLanguageModel:
    def __init__(self, discount=0.75):
        self.discount=discount
        self.max_n=5
        self.ngram={i:defaultdict(int) for i in range(1,6)}
        self.follow={i:defaultdict(set) for i in range(2,6)}
        self.precede=defaultdict(set)
        self.vocab=set()

    def train(self, corpus):
        for sent in corpus:
            words=["<s>"]*4+sent.lower().split()+["</s>"]
            self.vocab.update(words)
            for n in range(1,6):
                for i in range(len(words)-n+1):
                    g=tuple(words[i:i+n])
                    self.ngram[n][g]+=1
                    if n>1:
                        self.follow[n][g[:-1]].add(g[-1])
                        self.precede[g[-1]].add(g[:-1])

    def continuation_prob(self, word):
        total=sum(len(v) for v in self.precede.values())
        if total==0:return 1/max(1,len(self.vocab))
        return len(self.precede[word])/total if word in self.precede else 1/len(self.vocab)

    def kn(self, context, word):
        n=len(context)+1
        if n==1:
            return self.continuation_prob(word)
        gram=tuple(context+[word])
        ctx=tuple(context)
        c_ng=self.ngram[n].get(gram,0)
        c_ctx=self.ngram[n-1].get(ctx,0)
        if c_ctx==0:
            return self.kn(context[1:],word)
        first=max(c_ng-self.discount,0)/c_ctx
        lam=(self.discount*len(self.follow[n].get(ctx,set())))/c_ctx
        return first+lam*self.kn(context[1:],word)

    def predict(self, text, k=5):
        ctx = (["<s>"] * 4 + text.lower().split())[-4:]
        scores = {}
        for w in self.vocab:

        # Ignore start and end tokens
            if w == "<s>" or w == "</s>":
                
                continue
            scores[w] = self.kn(ctx, w)

        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    def update(self,sentence):
        self.train([sentence])

corpus=[
"best places to visit in india",
"best places to visit in chennai",
"best places to visit near me",
"best places to visit during summer",
"best places to visit with family",
"best restaurants near me",
"places to visit in kerala"
]

model=FiveGramLanguageModel()
model.train(corpus)

query = "best places to visit"

print("User Query:")
print(query)

print("\nAutocomplete Suggestions:\n")

for word, prob in model.predict(query):

    print(query + " " + word, "  Probability =", round(prob,4))

print("\nAdding new trending query...\n")

model.update("best places to visit in singapore")

print("Suggestions After Update:\n")

for word, prob in model.predict(query):

    print(query + " " + word, "  Probability =", round(prob,4))

User Query:
best places to visit

Autocomplete Suggestions:

best places to visit in   Probability = 0.6473
best places to visit near   Probability = 0.0994
best places to visit during   Probability = 0.0966
best places to visit with   Probability = 0.0966
best places to visit places   Probability = 0.0057

Adding new trending query...

Suggestions After Update:

best places to visit in   Probability = 0.751
best places to visit near   Probability = 0.0732
best places to visit during   Probability = 0.0718
best places to visit with   Probability = 0.0718
best places to visit places   Probability = 0.0028
